# Milestone 4: Two-tower retrieval

Starter notebook for milestone 4 of [Project - MovieLens Recommender](Project%20-%20MovieLens%20Recommender.md).

Cells marked **TODO** are yours to fill in; everything else runs as-is. Notebook is saved without outputs.

In [ ]:
import numpy as np, pandas as pd, pathlib, urllib.request, zipfile, io

DATA = pathlib.Path("ml-latest-small")
if not DATA.exists():
    # Or download manually from https://grouplens.org/datasets/movielens/ (ml-latest-small) and unzip here
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(".")
ratings = pd.read_csv(DATA / "ratings.csv")   # userId, movieId, rating, timestamp
movies = pd.read_csv(DATA / "movies.csv")     # movieId, title, genres
links = pd.read_csv(DATA / "links.csv")       # movieId, imdbId, tmdbId
ratings.head()

In [ ]:
# Leave-last-out split by time: for every user, the most recent positive is the test item,
# the one before it is validation. Positives = rating >= 4 (implicit feedback).
pos = ratings[ratings.rating >= 4].sort_values(["userId", "timestamp"])
pos["rank"] = pos.groupby("userId").cumcount(ascending=False)   # 0 = most recent
test  = pos[pos["rank"] == 0][["userId", "movieId"]]
val   = pos[pos["rank"] == 1][["userId", "movieId"]]
train = pos[pos["rank"] >= 2][["userId", "movieId"]]

# Contiguous ids so we can index embedding tables later
uid = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}
iid = {m: i for i, m in enumerate(sorted(movies.movieId.unique()))}   # includes never-rated movies (cold items)
n_users, n_items = len(uid), len(iid)
print(n_users, n_items, len(train), len(val), len(test))

In [ ]:
def evaluate(score_fn, held_out, train, k=10, n_neg=99, seed=0):
    """HitRate@k and NDCG@k with 1 held-out positive vs n_neg sampled unseen items (sampled protocol).
    score_fn(user_id, item_ids) -> array of scores. Switch to full ranking for final numbers,
    sampled metrics can reorder models (see Evaluating Recommenders)."""
    rng = np.random.default_rng(seed)
    seen = train.groupby("userId").movieId.apply(set).to_dict()
    all_items = np.array(list(iid))
    hits, ndcg = [], []
    for u, m in held_out.itertuples(index=False):
        banned = seen.get(u, set()) | {m}
        negs = rng.choice(all_items, size=n_neg * 2, replace=False)
        negs = [x for x in negs if x not in banned][:n_neg]
        cand = np.array([m] + negs)
        s = np.asarray(score_fn(u, cand))
        rank = (s > s[0]).sum() + rng.integers(0, (s == s[0]).sum())   # 0 = first; ties broken at random (a constant scorer must score ~chance)
        hits.append(rank < k)
        ndcg.append(1 / np.log2(rank + 2) if rank < k else 0.0)
    return {"HR@k": float(np.mean(hits)), "NDCG@k": float(np.mean(ndcg))}

## 1. Features
User tower: user id embedding + mean of the embeddings of the last 20 watched items. Item tower: item id embedding + genre multi-hot + release year (parse from the title).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
genres = sorted({g for gs in movies.genres for g in gs.split("|")})
G = np.zeros((n_items, len(genres)), dtype=np.float32)
for m, gs in zip(movies.movieId, movies.genres):
    for g in gs.split("|"):
        G[iid[m], genres.index(g)] = 1
G = torch.tensor(G)
# TODO: build `item_year` (regex r"\((\d{4})\)" on title, fill missing with median, standardise)

In [ ]:
class Tower(nn.Module):
    def __init__(self, d_in, d_out=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Linear(128, d_out))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)    # unit vectors: dot product = cosine

# TODO: define user_features(u) and item_features(i) and assemble the two towers

## 2. In-batch softmax loss
For a batch of B positive pairs, logits $= UI^\top/\tau$ ($B\times B$), labels = diagonal. Add the logQ correction ($\text{logits}_{uj}\mathrel{-}=\log q_j$) and compare with and without it. Mask accidental hits (same item twice in a batch).

In [ ]:
def in_batch_loss(u_emb, i_emb, tau=0.07, log_q=None):
    raise NotImplementedError  # TODO

## 3. Retrieval (TODO)
Precompute all item embeddings, retrieve top-K with a matrix product, then compare against an approximate index (FAISS or hnswlib) on recall@100 vs latency. Evaluate with full ranking, not sampled negatives.

In [ ]:
# TODO